# Session 1B: The Vanilla Agent
**Goal**: Build a ReAct (Reason+Act) loop from scratch to let the LLM call real functions.

In [ ]:
import os
import sys
import json
sys.path.insert(0, '..')
import utils.bootstrap

import google.generativeai as genai
from utils.gmail_utils import fetch_recent_emails
from utils.calendar_utils import create_calendar_event
from utils.tools import format_emails

genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

### Step 1: Define the Tools (Real Python Functions)

In [ ]:
def tool_fetch_emails(**kwargs):
    limit = int(kwargs.get('limit', 5))
    return format_emails(fetch_recent_emails(limit=limit))

def tool_schedule_meeting(**kwargs):
    time = kwargs.get('time', '')
    attendees_raw = kwargs.get('attendees', [])
    title = kwargs.get('title', 'AI Scheduled Meeting')
    attendees = [a.strip() for a in attendees_raw.split(',')] if isinstance(attendees_raw, str) else list(attendees_raw)
    try:
        link = create_calendar_event(time, attendees, title)
        if 'already scheduled' in link: return link
        return f"Meeting '{title}' scheduled at {time}. Link: {link}"
    except ValueError as e:
        return f"Error: {str(e)}"

TOOLS = {
    'fetch_emails': {'function': tool_fetch_emails},
    'schedule_meeting': {'function': tool_schedule_meeting}
}

### Step 2: Set the System Prompt & Init Loop

In [ ]:
SYSTEM_PROMPT = """You are an AI email assistant.\nYou have access to these tools:\n1. fetch_emails(limit)\n2. schedule_meeting(time, attendees, title)\n\nRULES:\nTo call a tool, output EXACTLY ONE JSON object: {"tool": "name", "args": {}}\nWhen done, output: {"tool": "DONE", "summary": "..."}"""

model = genai.GenerativeModel('gemini-flash-latest')
conversation = [
    {"role": "user", "parts": [SYSTEM_PROMPT]},
    {"role": "model", "parts": ['{"tool": "acknowledge", "status": "ready"}']},
    {"role": "user", "parts": ["Fetch my emails, analyze them, and schedule any meetings."]}
]

### Step 3: Run the Agent Loop

In [ ]:
import re
for i in range(1, 6):
    print(f"\n--- Iteration {i} ---")
    response = model.generate_content(conversation)
    raw = response.text.strip()
    print(f"LLM says: {raw[:200]}...")
    
    # Parse JSON
    match = re.search(r'\{[^{}]*\}', raw)
    if not match: break
    action = json.loads(match.group())
    
    tool_name = action.get('tool')
    if tool_name == 'DONE':
        print("\n\u2705 FINISHED!", action.get('summary'))
        break
        
    print(f"\ud83d\udd27 Calling {tool_name}...")
    args = action.get('args', {})
    result = TOOLS[tool_name]['function'](**args)
    print(f"\ud83d\udce6 Result: {str(result)[:100]}...")
    
    # Feedback loop
    conversation.append({"role": "model", "parts": [raw]})
    conversation.append({"role": "user", "parts": [f"Tool returned: {result}\nNext action? (Reply in JSON)"]})